### 1. Database bağlantısının hazırlanması

Bu mərhələdə notebook SQL Server ilə işləməyə hazırlanır. Lazım olan import-lar çağırılır və database connection yaradılır.


In [2]:
from common import *

import common
from db import get_engine

engine = get_engine()

### 1.Qısa nəticə:
Database ilə işləmək üçün connection hazırlandı.

### 2. Silver cədvəlinin yoxlanılması

Bu mərhələdə `silver.flights_clean` cədvəlinin SQL Server-də mövcud olduğu və neçə sətirdən ibarət olduğu yoxlanılır.

In [ ]:
import pandas as pd

pd.read_sql("""
SELECT COUNT(*) AS total_rows
FROM silver.flights_clean
""", con=engine)

### 2.Qısa nəticə:

`silver.flights_clean` cədvəlinin SQL Server-də mövcud olduğu və analiz üçün əlçatan olduğu yoxlanıldı.

## 3. Silver datasının ilkin baxışı 
Bu mərhələdə analiz edəcəyimiz silver.flights_clean cədvəlinin ilk sətirlərinə baxılır. Məqsəd sütunların və məlumatların düzgün göründüyünü yoxlamaqdır.

In [4]:
pd.read_sql("""SELECT TOP 5 *
FROM silver.flights_clean""", con=engine)

,MONTH,DAY,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DISTANCE,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL,DEPARTURE_DELAY,...,WEATHER_SURFACE_PRESSURE,WEATHER_WIND_DIRECTION,WEATHER_SNOWFALL,WEATHER_WIND_GUSTS,YEAR,IS_DELAYED,FLIGHT_STATUS,IS_WEEKEND,DEPARTURE_PERIOD,IS_LONG_DISTANCE
0,June,29,1,EV,ATL,TRI,227,12:05:00,13:08:00,-4.0,...,980.900024,310.0,0.0,14.8,2015,0,Completed,0,Afternoon,0
1,April,19,7,DL,ATL,PNS,271,17:35:00,17:48:00,-2.0,...,975.000000,193.0,0.0,47.5,2015,0,Completed,1,Afternoon,0
2,August,18,2,WN,RDU,STL,667,18:50:00,19:55:00,296.0,...,NaN,NaN,NaN,NaN,2015,1,Completed,0,Evening,0
3,August,16,7,HA,KOA,HNL,163,13:41:00,14:25:00,-3.0,...,NaN,NaN,NaN,NaN,2015,0,Completed,1,Afternoon,0
4,July,30,4,WN,LAX,SJC,308,19:45:00,20:50:00,102.0,...,1010.700012,289.0,0.0,19.1,2015,1,Completed,0,Evening,0


### 4. Ümumi uçuş vəziyyətinin yoxlanılması

Bu mərhələdə uçuşların neçə dənəsinin tamamlandığı, ləğv edildiyi və başqa istiqamətə yönləndirildiyi yoxlanılır.

In [ ]:
pd.read_sql("""
SELECT
    FLIGHT_STATUS,
    COUNT(*) AS FLIGHT_COUNT
FROM silver.flights_clean
GROUP BY FLIGHT_STATUS
ORDER BY FLIGHT_COUNT DESC
""", con=engine)

,FLIGHT_STATUS,FLIGHT_COUNT
0,Completed,981515
1,Cancelled,15527
2,Diverted,2615


### 4.Qısa nəticə:

Analiz nəticəsində uçuşların böyük hissəsinin uğurla tamamlandığı görüldü. Ümumilikdə **981,515 uçuş Completed**, **15,527 uçuş Cancelled**, **2,615 uçuş isə Diverted** olub.

Bu nəticə uçuş əməliyyatlarının ümumi vəziyyətini göstərir və əsas problemlərin daha çox hansı istiqamətlərdə olduğunu araşdırmaq üçün ilkin baza yaradır.


### 5. Aviaşirkətlər üzrə uçuş performansının müqayisəsi

Bu mərhələdə aviaşirkətlərin uçuş performansı müqayisə olunur. Məqsəd hansı aviaşirkətlərdə gecikmə faizinin və orta çatma gecikməsinin daha yüksək və ya aşağı olduğunu görməkdir.

In [6]:
pd.read_sql("""
SELECT
    AIRLINE_NAME,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY AIRLINE_NAME
ORDER BY DELAY_RATE
""", con=engine)

,AIRLINE_NAME,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,Hawaiian Airlines Inc.,13122,10.93,1.87
1,Alaska Airlines Inc.,29609,13.05,-1.06
2,Delta Air Lines Inc.,149858,13.65,0.28
3,American Airlines Inc.,124964,18.20,3.18
4,Skywest Airlines Inc.,100614,18.63,5.83
5,US Airways Inc.,34278,18.89,3.77
6,Southwest Airlines Co.,217546,19.04,4.39
7,Virgin America,10703,19.35,4.91
8,Atlantic Southeast Airlines,98165,19.64,6.66
9,United Air Lines Inc.,88596,20.54,5.33


### 5.Qısa nəticə:

Aviaşirkətlər üzrə müqayisədə ən aşağı gecikmə faizi **Hawaiian Airlines Inc.** şirkətində olub (**10.93%**).

Ən yüksək gecikmə faizi isə **Spirit Air Lines** şirkətində müşahidə olunub (**29.93%**). Spirit Air Lines həmçinin orta hesabla **14.23 dəqiqə** çatma gecikməsi ilə ən zəif nəticələrdən birini göstərib.

**Alaska Airlines Inc.** üçün orta çatma gecikməsinin **-1.06 dəqiqə** olması uçuşların orta hesabla planlaşdırılmış vaxtdan bir qədər tez çatdığını göstərir.

Nəticə olaraq, aviaşirkətlər arasında uçuş performansında nəzərəçarpacaq fərqlər olduğu görünür.


### 6. Başlanğıc hava limanlarında yerüstü əməliyyatların müqayisəsi

Bu mərhələdə hava limanları üzrə orta `TAXI_OUT` vaxtı müqayisə olunur. Məqsəd təyyarələrin uçuşdan əvvəl ən çox vaxt itirdiyi hava limanlarını müəyyən etməkdir.

In [7]:
pd.read_sql("""
SELECT TOP 10
    ORIGIN_AIRPORT,
    ORIGIN_AIRPORT_NAME,
    ROUND(AVG(TAXI_OUT), 2) AS AVG_TAXI_OUT
FROM silver.flights_clean
GROUP BY ORIGIN_AIRPORT, ORIGIN_AIRPORT_NAME
ORDER BY AVG_TAXI_OUT DESC
""", con=engine)

,ORIGIN_AIRPORT,ORIGIN_AIRPORT_NAME,AVG_TAXI_OUT
0,LGA,LaGuardia Airport (Marine Air Terminal),26.84
1,JFK,John F. Kennedy International Airport (New Yor...,26.67
2,MMH,Mammoth Yosemite Airport,25.68
3,PHL,Philadelphia International Airport,21.47
4,ISN,Sloulin Field International Airport,20.38
5,INL,Falls International Airport,20.20
6,BRD,Brainerd Lakes Regional Airport,20.11
7,EWR,Newark Liberty International Airport,20.08
8,BOS,Gen. Edward Lawrence Logan International Airport,19.99
9,IMT,Ford Airport,19.75


### 6.Qısa nəticə:

Hava limanları üzrə müqayisədə ən yüksək orta `TAXI_OUT` vaxtı **LaGuardia Airport (LGA)** hava limanında müşahidə olunub — **26.84 dəqiqə**.

İkinci yerdə **John F. Kennedy International Airport (JFK)** **26.67 dəqiqə**, üçüncü yerdə isə **Mammoth Yosemite Airport (MMH)** **25.68 dəqiqə** ilə qərarlaşıb.

Bu nəticə həmin hava limanlarında təyyarələrin uçuşdan əvvəl yerüstü mərhələdə daha çox vaxt keçirdiyini göstərir.

Lakin yalnız orta `TAXI_OUT` vaxtına əsasən hava limanının zəif işlədiyini demək olmaz. Uçuş sayı və hava limanının sıxlığı da nəzərə alınmalıdır.


### 7. Marşrutlar üzrə uçuş performansının müqayisəsi

Bu mərhələdə başlanğıc və təyinat hava limanlarına görə marşrutların gecikmə faizi müqayisə olunur. Məqsəd daha çox problem yaşanan marşrutları müəyyən etməkdir.

In [8]:
pd.read_sql("""
SELECT TOP 10
    ORIGIN_AIRPORT,
    DESTINATION_AIRPORT,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE
FROM silver.flights_clean
GROUP BY ORIGIN_AIRPORT, DESTINATION_AIRPORT
HAVING COUNT(*) >= 500
ORDER BY DELAY_RATE DESC
""", con=engine)

,ORIGIN_AIRPORT,DESTINATION_AIRPORT,TOTAL_FLIGHTS,DELAY_RATE
0,ORD,DEN,1036,32.29
1,PBI,LGA,564,30.60
2,IAH,LAX,826,29.58
3,LGA,PBI,515,29.45
4,ORD,SFO,1418,28.77
5,ORD,IAH,852,28.74
6,LAS,LAX,1798,28.34
7,SJC,LAX,1207,28.11
8,LGA,FLL,991,28.07
9,OAK,LAX,1202,27.96


### 7.Qısa nəticə:

Marşrutlar üzrə müqayisədə ən yüksək gecikmə faizi **ORD → DEN** marşrutunda müşahidə olunub — **32.29%**. Bu marşrut üzrə ümumilikdə **1,036 uçuş** həyata keçirilib.

İkinci yerdə **PBI → LGA** marşrutu **30.60%**, üçüncü yerdə isə **IAH → LAX** marşrutu **29.58%** gecikmə faizi ilə qərarlaşıb.

Nəticə göstərir ki, bəzi marşrutlarda gecikmə riski digərlərinə nisbətən daha yüksəkdir. Analizdə yalnız ən azı **500 uçuşu olan marşrutlar** nəzərə alındığı üçün nəticələr daha etibarlı müqayisə verir.


### 8. Aylar üzrə uçuş performansının müqayisəsi

Bu mərhələdə uçuşların aylara görə performansı müqayisə olunur. Məqsəd hansı aylarda gecikmə faizinin daha yüksək olduğunu müəyyən etməkdir.

In [9]:
pd.read_sql("""
SELECT
    MONTH,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY,
    ROUND(100.0 * SUM(CANCELLED) / COUNT(*), 2) AS CANCELLATION_RATE,
    ROUND(100.0 * SUM(DIVERTED) / COUNT(*), 2) AS DIVERSION_RATE
FROM silver.flights_clean
GROUP BY MONTH
ORDER BY DELAY_RATE DESC
""", con=engine)

,MONTH,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY,CANCELLATION_RATE,DIVERSION_RATE
0,June,86385,23.52,9.58,1.82,0.39
1,February,74041,23.16,8.28,4.75,0.23
2,January,80869,21.01,5.70,2.57,0.22
3,July,89974,20.67,6.21,0.95,0.29
4,December,82536,20.64,6.07,1.75,0.30
5,March,86649,19.41,5.01,2.15,0.23
6,August,87623,18.60,4.51,1.01,0.30
7,May,85504,18.26,4.45,1.16,0.34
8,April,83792,17.30,3.43,0.93,0.30
9,November,80361,15.17,0.88,1.00,0.22


### 8.Qısa nəticə:

Aylar üzrə müqayisədə ən yüksək gecikmə faizi **June ayında 23.52%**, ikinci ən yüksək göstərici isə **February ayında 23.16%** olub.

**February** ayında cancellation rate **4.75%** ilə digər aylardan daha yüksək olub.

Ən aşağı gecikmə faizi isə **October ayında 12.65%** müşahidə olunub.

Nəticə olaraq, uçuş performansının aylara görə dəyişdiyi və xüsusilə **June, February və January** aylarında daha zəif olduğu görünür.


### 9. Həftəiçi və həftəsonu uçuş performansının müqayisəsi

Bu mərhələdə həftəiçi və həftəsonu uçuş performansı müqayisə olunur. Məqsəd gecikmələrin hansı gün qrupunda daha çox olduğunu müəyyən etməkdir.

In [10]:
pd.read_sql("""
SELECT
    IS_WEEKEND,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY IS_WEEKEND
ORDER BY DELAY_RATE DESC
""", con=engine)

,IS_WEEKEND,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,0,738808,19.10,4.92
1,1,260849,17.16,2.88


### 9.Qısa nəticə:

Həftəiçi uçuşlarda gecikmə faizi **19.10%**, həftəsonu uçuşlarda isə **17.16%** olub.

Orta çatma gecikməsi də həftəiçi uçuşlarda daha yüksəkdir: **4.92 dəqiqə**, həftəsonunda isə **2.88 dəqiqə**.

Nəticə olaraq, həftəiçi uçuş performansının həftəsonuna nisbətən bir qədər zəif olduğu görünür.


## 10. Günün vaxtına görə uçuş performansının müqayisəsi

Bu mərhələdə uçuş performansı günün müxtəlif vaxtlarına görə müqayisə olunur. Məqsəd gecikmələrin daha çox hansı zaman aralığında baş verdiyini müəyyən etməkdir.

In [11]:
pd.read_sql("""
SELECT
    DEPARTURE_PERIOD,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY DEPARTURE_PERIOD
ORDER BY DELAY_RATE DESC
""", con=engine)

,DEPARTURE_PERIOD,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,Evening,218254,25.34,9.02
1,Afternoon,368276,21.06,6.56
2,Morning,388979,13.09,0.21
3,Night,24148,9.09,-3.14


### 10.Qısa nəticə:

Günün vaxtına görə müqayisədə ən yüksək gecikmə faizi **Evening** dövründə müşahidə olunub — **25.34%**. Bu dövrdə orta çatma gecikməsi **9.02 dəqiqə** olub.

**Afternoon** dövründə gecikmə faizi **21.06%**, **Morning** dövründə isə **13.09%** olub.

Ən aşağı gecikmə faizi **Night** dövründə qeydə alınıb — **9.09%**. Night uçuşlarında orta çatma gecikməsinin **-3.14 dəqiqə** olması uçuşların orta hesabla planlaşdırılmış vaxtdan daha tez çatdığını göstərir.

Nəticə olaraq, uçuş performansının günün saatına görə dəyişdiyi və xüsusilə axşam saatlarında daha zəif olduğu görünür.


## 11. Uçuş gecikmələrinin əsas səbəblərinin analizi

Bu mərhələdə gecikmiş uçuşların əsas səbəbləri müqayisə olunur. Məqsəd gecikmələrin daha çox aviaşirkət, gec gələn təyyarə, hava sistemi, hava şəraiti və ya təhlükəsizlik səbəbindən yarandığını müəyyən etməkdir.

In [12]:
pd.read_sql("""
SELECT
    ROUND(AVG(AIRLINE_DELAY), 2) AS AIRLINE_DELAY,
    ROUND(AVG(LATE_AIRCRAFT_DELAY), 2) AS LATE_AIRCRAFT_DELAY,
    ROUND(AVG(AIR_SYSTEM_DELAY), 2) AS AIR_SYSTEM_DELAY,
    ROUND(AVG(WEATHER_DELAY), 2) AS WEATHER_DELAY,
    ROUND(AVG(SECURITY_DELAY), 2) AS SECURITY_DELAY
FROM silver.flights_clean
WHERE IS_DELAYED = 1
""", con=engine)

,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,AIR_SYSTEM_DELAY,WEATHER_DELAY,SECURITY_DELAY
0,18.92,23.38,13.49,2.97,0.07


### 11.Qısa nəticə:

Gecikmiş uçuşlarda ən yüksək orta gecikmə **Late Aircraft Delay** səbəbindən yaranıb — **23.38 dəqiqə**.

İkinci əsas səbəb **Airline Delay** olub — **18.92 dəqiqə**. Daha sonra **Air System Delay** orta hesabla **13.49 dəqiqə** təsir göstərib.

**Weather Delay** orta hesabla **2.97 dəqiqə**, **Security Delay** isə cəmi **0.07 dəqiqə** olub.

Nəticə olaraq, gecikmələrdə əsas yük daha çox əvvəlki təyyarənin gec gəlməsi və aviaşirkətlə bağlı səbəblərin üzərinə düşür.


## 12. Hava şəraitinin uçuş performansına təsiri

Bu mərhələdə yağıntılı və yağıntısız hava şəraitində uçuş performansı müqayisə olunur. Məqsəd hava şəraitinin gecikmə və orta çatma vaxtına təsir edib-etmədiyini görməkdir.

In [14]:
pd.read_sql("""
SELECT
    CASE WHEN WEATHER_PRECIPITATION > 0 THEN 'Rain' ELSE 'No Rain' END AS WEATHER_GROUP,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
WHERE WEATHER_PRECIPITATION IS NOT NULL
GROUP BY CASE WHEN WEATHER_PRECIPITATION > 0 THEN 'Rain' ELSE 'No Rain' END
""", con=engine)

,WEATHER_GROUP,DELAY_RATE,AVG_ARRIVAL_DELAY
0,Rain,28.29,13.78
1,No Rain,17.30,3.16


### 12.Qısa nəticə:

Yağıntılı hava şəraitində gecikmə faizi **28.29%**, yağıntısız havada isə **17.30%** olub.

Orta çatma gecikməsi də yağıntılı havada daha yüksəkdir: **13.78 dəqiqə**, yağıntısız havada isə **3.16 dəqiqə**.

Nəticə olaraq, yağıntılı hava şəraitində uçuş performansının daha zəif olduğu görünür.


## 13. Uçuş məsafəsinin performansa təsiri

Bu mərhələdə qısa və uzun məsafəli uçuşların performansı müqayisə olunur. Məqsəd uçuş məsafəsinin gecikmə faizi və orta çatma gecikməsinə təsir edib-etmədiyini görməkdir.

In [15]:
pd.read_sql("""
SELECT
    IS_LONG_DISTANCE,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY,
    ROUND(AVG(AIR_TIME), 2) AS AVG_AIR_TIME
FROM silver.flights_clean
GROUP BY IS_LONG_DISTANCE
""", con=engine)

,IS_LONG_DISTANCE,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY,AVG_AIR_TIME
0,0,864207,18.62,4.79,90.97
1,1,135450,18.43,1.84,256.54


## 13.Qısa nəticə:

Qısa və orta məsafəli uçuşlarda gecikmə faizi **18.62%**, uzun məsafəli uçuşlarda isə **18.43%** olub. Gecikmə faizləri bir-birinə çox yaxın olduğu üçün məsafənin gecikmə ehtimalına ciddi fərq yaratmadığı görünür.

Orta çatma gecikməsi qısa və orta məsafəli uçuşlarda **4.79 dəqiqə**, uzun məsafəli uçuşlarda isə **1.84 dəqiqə** olub.

Uzun məsafəli uçuşların orta hava vaxtı **256.54 dəqiqə**, digər uçuşlarda isə **90.97 dəqiqə** təşkil edib.

Nəticə olaraq, uzun məsafəli uçuşlar havada daha çox vaxt keçirsə də, gecikmə faizi baxımından ciddi fərq müşahidə olunmur.


## 14. Uçuş zamanı gecikmənin kompensasiya olunmasının analizi

Bu mərhələdə gec başlayan uçuşların uçuş zamanı gecikmənin bir hissəsini kompensasiya edib-etmədiyi yoxlanılır. Məqsəd hansı aviaşirkətlərin gecikməni daha yaxşı azaltdığını müəyyən etməkdir.

In [16]:
pd.read_sql("""
SELECT TOP 10
    AIRLINE_NAME,
    ROUND(AVG(DEPARTURE_DELAY), 2) AS AVG_DEPARTURE_DELAY,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY,
    ROUND(AVG(DEPARTURE_DELAY - ARRIVAL_DELAY), 2) AS TIME_RECOVERED
FROM silver.flights_clean
WHERE DEPARTURE_DELAY > 0
  AND ARRIVAL_DELAY IS NOT NULL
GROUP BY AIRLINE_NAME
ORDER BY TIME_RECOVERED DESC
""", con=engine)

,AIRLINE_NAME,AVG_DEPARTURE_DELAY,AVG_ARRIVAL_DELAY,TIME_RECOVERED
0,United Air Lines Inc.,32.38,23.52,8.86
1,Delta Air Lines Inc.,29.89,22.99,6.90
2,Southwest Airlines Co.,26.87,21.20,5.67
3,American Airlines Inc.,33.50,28.45,5.05
4,JetBlue Airways,37.20,32.99,4.21
5,Virgin America,29.72,25.59,4.13
6,Alaska Airlines Inc.,25.86,22.39,3.47
7,Skywest Airlines Inc.,38.94,36.77,2.18
8,US Airways Inc.,28.49,26.46,2.03
9,Atlantic Southeast Airlines,41.17,39.62,1.54


### 14.Qısa nəticə:

Gec başlayan uçuşlarda gecikməni ən çox kompensasiya edən aviaşirkət **United Air Lines Inc.** olub. Bu şirkət orta hesabla **8.86 dəqiqə** gecikməni uçuş zamanı azaldıb.

İkinci yerdə **Delta Air Lines Inc.** **6.90 dəqiqə**, üçüncü yerdə isə **Southwest Airlines Co.** **5.67 dəqiqə** gecikmə kompensasiyası ilə qərarlaşıb.

Nəticə göstərir ki, bəzi aviaşirkətlər gec başlayan uçuşlarda vaxtın bir hissəsini uçuş zamanı daha yaxşı bərpa edə bilir.


## 15. Axşam uçuşlarında aviaşirkətlər üzrə dərin analiz

Əvvəlki analizdə ən yüksək gecikmə faizinin axşam uçuşlarında olduğu müəyyən edildi. Bu mərhələdə axşam saatlarında hansı aviaşirkətlərin daha yüksək gecikmə göstəricisinə malik olduğu araşdırılır.

In [17]:
pd.read_sql("""
SELECT TOP 10
    AIRLINE_NAME,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
WHERE DEPARTURE_PERIOD = 'Evening'
GROUP BY AIRLINE_NAME
HAVING COUNT(*) >= 1000
ORDER BY DELAY_RATE DESC
""", con=engine)

,AIRLINE_NAME,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,Spirit Air Lines,5929,36.60,19.76
1,JetBlue Airways,12212,32.50,14.87
2,Frontier Airlines Inc.,4530,31.87,16.37
3,Southwest Airlines Co.,49396,30.78,12.78
4,United Air Lines Inc.,18460,25.83,9.18
5,Atlantic Southeast Airlines,18263,24.98,9.76
6,American Eagle Airlines Inc.,10700,24.51,8.63
7,US Airways Inc.,7768,24.50,7.79
8,Virgin America,2168,24.02,8.63
9,American Airlines Inc.,26957,23.65,6.77


### 15.Qısa nəticə:

Axşam uçuşlarında ən yüksək gecikmə faizi **Spirit Air Lines** şirkətində müşahidə olunub — **36.60%**. Bu şirkətdə orta çatma gecikməsi də **19.76 dəqiqə** olub.

İkinci yerdə **JetBlue Airways** **32.50%**, üçüncü yerdə isə **Frontier Airlines Inc.** **31.87%** gecikmə faizi ilə qərarlaşıb.

Nəticə göstərir ki, əvvəlki analizdə problemli görünən **Evening** dövründə gecikmə xüsusilə bəzi aviaşirkətlərdə daha yüksəkdir.


## 16. Ümumi uçuş performansının yekun KPI-ları

Bu mərhələdə layihənin əsas uçuş göstəriciləri bir cədvəldə ümumiləşdirilir. Məqsəd ümumi uçuş performansını qısa şəkildə göstərməkdir.

In [18]:
pd.read_sql("""
SELECT
    COUNT(*) AS TOTAL_FLIGHTS,
    SUM(CASE WHEN FLIGHT_STATUS = 'Completed' THEN 1 ELSE 0 END) AS COMPLETED_FLIGHTS,
    SUM(CANCELLED) AS CANCELLED_FLIGHTS,
    SUM(DIVERTED) AS DIVERTED_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
""", con=engine)

,TOTAL_FLIGHTS,COMPLETED_FLIGHTS,CANCELLED_FLIGHTS,DIVERTED_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,999657,981515,15527,2615,18.6,4.39


### 16.Qısa nəticə:

Dataset üzrə ümumilikdə **999,657 uçuş** analiz edilib.

Onlardan **981,515-i tamamlanıb**, **15,527-si ləğv edilib**, **2,615-i isə başqa istiqamətə yönləndirilib**.

Ümumi gecikmə faizi **18.60%**, orta çatma gecikməsi isə **4.39 dəqiqə** olub.

Nəticə olaraq, uçuşların böyük əksəriyyəti uğurla tamamlanıb, lakin təxminən hər 5 uçuşdan 1-də gecikmə müşahidə olunub.
